In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [2]:
df = pd.read_csv(
    "../data/raw/istanbul_apartment_prices_2026.csv"
)

print("Dataset shape:", df.shape)

df.head()

Dataset shape: (24767, 30)


,listing_id,price,price_per_sqm,district,neighborhood,rooms,halls,total_rooms,gross_sqm,net_sqm,...,usage_status,is_in_complex,complex_name,orientation,maintenance_fee,credit_eligible,deed_status,exchange,last_updated,scraped_at
0,54895-1121,15900000,165625.00,Adalar,Maden,3,1,4,138.0,96.0,...,Vacant,0,NaN,"North, South, East, West",2500.0,Not Eligible,Land Title,No,2026-03-11,2026-03-14T20:38:29
1,54895-1103,40000000,266666.67,Adalar,Nizam,4,1,5,199.0,150.0,...,Vacant,1,NaN,"North, South, East, West",10000.0,Eligible,Construction Easement,No,2026-03-05,2026-03-14T20:38:36
2,54895-1148,17850000,220370.37,Adalar,Nizam,2,1,3,107.0,81.0,...,Vacant,0,NaN,"North, East, West",1000.0,Eligible,Condominium Title,No,2026-03-05,2026-03-14T20:38:47
3,12854-258,16000000,81632.65,Adalar,Heybeliada,3,1,4,200.0,196.0,...,Owner-occupied,0,NaN,"North, West",NaN,Eligible,Condominium Title,No,2026-03-14,2026-03-14T20:38:56
4,121334-99,14000000,96551.72,Adalar,Nizam,4,1,5,160.0,145.0,...,Vacant,0,NaN,NaN,400.0,Eligible,Condominium Title,No,2026-03-14,2026-03-14T20:39:04


In [3]:
df_clean = df.copy()

print("Original:", df.shape)
print("Working copy:", df_clean.shape)

Original: (24767, 30)
Working copy: (24767, 30)


In [4]:
low_price_mask = df_clean["price"] < 1_000_000
extreme_price_mask = df_clean["price"] > 500_000_000

print("1M TL altı:", low_price_mask.sum())
print("500M TL üstü:", extreme_price_mask.sum())

1M TL altı: 30
500M TL üstü: 9


In [5]:
df_clean["price"] < 1_000_000

0        False
1        False
2        False
3        False
4        False
         ...  
24762    False
24763    False
24764    False
24765    False
24766    False
Name: price, Length: 24767, dtype: bool

In [6]:
district_median_sqm = (
    df_clean[
        df_clean["price"].between(1_000_000, 500_000_000)
    ]
    .groupby("district")["price_per_sqm"]
    .median()
    .sort_values()
)

district_median_sqm

district
Esenyurt          32550.775
Sultangazi        43333.330
Beylikdüzü        44726.000
Silivri           46733.335
Esenler           47220.590
Arnavutköy        48166.670
Avcılar           51082.355
Fatih             51449.580
Gaziosmanpaşa     53660.715
Büyükçekmece      56250.000
Bağcılar          56619.050
Bayrampaşa        57071.430
Sultanbeyli       58200.000
Küçükçekmece      58288.770
Sancaktepe        59705.880
Güngören          60376.675
Çatalca           60769.230
Kağıthane         68062.500
Çekmeköy          70000.000
Bahçelievler      71232.880
Başakşehir        72727.270
Eyüpsultan        73076.920
Tuzla             78358.210
Pendik            80941.175
Şişli             82041.665
Ümraniye          83157.890
Ataşehir          94462.760
Kartal            95521.395
Zeytinburnu       96130.950
Beyoğlu          104501.920
Maltepe          108500.000
Üsküdar          114130.430
Beykoz           117321.430
Adalar           118181.820
Bakırköy         146616.540
Sarıyer    

In [7]:
extreme_check = df_clean[
    df_clean["price"] > 500_000_000
][
    [
        "price",
        "district",
        "neighborhood",
        "net_sqm",
        "price_per_sqm"
    ]
].copy()

extreme_check["district_median_sqm"] = (
    extreme_check["district"].map(district_median_sqm)
)

extreme_check["sqm_ratio"] = (
    extreme_check["price_per_sqm"]
    / extreme_check["district_median_sqm"]
)

extreme_check

,price,district,neighborhood,net_sqm,price_per_sqm,district_median_sqm,sqm_ratio
3566,8950000000,Bakırköy,Zuhuratbaba,75.0,1.193333e+08,146616.540,813.914537
5215,11500000000,Bayrampaşa,Kartaltepe,190.0,6.052632e+07,57071.430,1060.536170
11705,8750000000,Fatih,Şehremini,145.0,6.034483e+07,51449.580,1172.892521
12151,4500000000,Gaziosmanpaşa,Merkez,90.0,5.000000e+07,53660.715,931.780354
12453,4200000000,Güngören,Merkez,120.0,3.500000e+07,60376.675,579.694062
13791,3550000000,Kağıthane,Nurtepe,120.0,2.958333e+07,68062.500,434.649526
14561,10500000000,Kağıthane,Yahya Kemal,73.0,1.438356e+08,68062.500,2113.287294
16996,3825000000,Maltepe,Zümrütevler,60.0,6.375000e+07,108500.000,587.557604
20285,5189000000,Şişli,Kuştepe,150.0,3.459333e+07,82041.665,421.655671


In [8]:
price_per_sqm_q001 = df_clean["price_per_sqm"].quantile(0.001)
price_per_sqm_q999 = df_clean["price_per_sqm"].quantile(0.999)

print("0.1 percentile:", price_per_sqm_q001)
print("99.9 percentile:", price_per_sqm_q999)

0.1 percentile: 11058.67934
99.9 percentile: 793014.1318400027


In [9]:
sqm_outlier_mask = (
    (df_clean["price_per_sqm"] < price_per_sqm_q001)
    |
    (df_clean["price_per_sqm"] > price_per_sqm_q999)
)

print("Price/m² outliers:", sqm_outlier_mask.sum())

Price/m² outliers: 50


In [10]:
area_error_mask = (
    df_clean["net_sqm"] > df_clean["gross_sqm"]
)

print("Area errors:", area_error_mask.sum())

Area errors: 5


In [11]:
quality_report = pd.DataFrame({
    "issue": [
        "Price below 1M TL",
        "Price above 500M TL",
        "Extreme price per sqm",
        "Net sqm greater than gross sqm"
    ],
    "count": [
        (df_clean["price"] < 1_000_000).sum(),
        (df_clean["price"] > 500_000_000).sum(),
        sqm_outlier_mask.sum(),
        area_error_mask.sum()
    ]
})

quality_report

,issue,count
0,Price below 1M TL,30
1,Price above 500M TL,9
2,Extreme price per sqm,50
3,Net sqm greater than gross sqm,5


In [12]:
confirmed_error_mask = (
    (df_clean["price"] > 500_000_000)
    |
    (df_clean["net_sqm"] > df_clean["gross_sqm"])
)

print("Confirmed erroneous rows:", confirmed_error_mask.sum())

Confirmed erroneous rows: 14


In [13]:
df_clean.loc[
    confirmed_error_mask,
    [
        "listing_id",
        "price",
        "district",
        "neighborhood",
        "gross_sqm",
        "net_sqm",
        "price_per_sqm"
    ]
].sort_values("price")

,listing_id,price,district,neighborhood,gross_sqm,net_sqm,price_per_sqm
23926,2118-10412,14800000,Üsküdar,Küçüksu,2.0,160.0,9.250000e+04
24729,91261-990,15200000,Zeytinburnu,Merkezefendi,1.0,110.0,1.381818e+05
19301,9636-16356,325000000,Sarıyer,Yeniköy,1.0,800.0,4.062500e+05
6131,2525-16202,339750000,Beşiktaş,Kuruçeşme,1.0,900.0,3.775000e+05
5945,123075-4642,339750000,Beşiktaş,Kuruçeşme,1.0,900.0,3.775000e+05
13791,109839-598,3550000000,Kağıthane,Nurtepe,130.0,120.0,2.958333e+07
16996,166730-3,3825000000,Maltepe,Zümrütevler,70.0,60.0,6.375000e+07
12453,159000-50,4200000000,Güngören,Merkez,140.0,120.0,3.500000e+07
12151,0-45938460,4500000000,Gaziosmanpaşa,Merkez,95.0,90.0,5.000000e+07
20285,144871-574,5189000000,Şişli,Kuştepe,160.0,150.0,3.459333e+07


In [14]:
rows_before = len(df_clean)

df_clean = df_clean.loc[~confirmed_error_mask].copy()

rows_after = len(df_clean)

print("Rows before cleaning:", rows_before)
print("Rows after cleaning:", rows_after)
print("Removed rows:", rows_before - rows_after)

Rows before cleaning: 24767
Rows after cleaning: 24753
Removed rows: 14


In [15]:
print(
    "Price above 500M:",
    (df_clean["price"] > 500_000_000).sum()
)

print(
    "Net > Gross:",
    (df_clean["net_sqm"] > df_clean["gross_sqm"]).sum()
)

Price above 500M: 0
Net > Gross: 0


In [16]:
missing_report = pd.DataFrame({
    "missing_count": df_clean.isnull().sum(),
    "missing_percent": (
        df_clean.isnull().mean() * 100
    ).round(2)
})

missing_report = missing_report[
    missing_report["missing_count"] > 0
].sort_values(
    "missing_percent",
    ascending=False
)

missing_report

,missing_count,missing_percent
complex_name,21001,84.84
floor_category,17364,70.15
maintenance_fee,15756,63.65
building_type,14557,58.81
fuel_type,13849,55.95
building_condition,13842,55.92
last_updated,11886,48.02
orientation,5310,21.45
floor,1915,7.74
exchange,1595,6.44


In [17]:
pd.crosstab(
    df_clean["is_in_complex"],
    df_clean["complex_name"].isna(),
    margins=True
)

complex_name,False,True,All
is_in_complex,,,
0,0,19209,19209
1,3752,1792,5544
All,3752,21001,24753


In [18]:
df_clean.groupby("is_in_complex")[
    "complex_name"
].apply(
    lambda x: x.isna().mean() * 100
)

is_in_complex
0    100.000000
1     32.323232
Name: complex_name, dtype: float64

In [19]:
df_clean[
    ["floor", "floor_category", "total_floors"]
].head(30)

,floor,floor_category,total_floors
0,3.0,NaN,3.0
1,-1.0,Garden Floor,3.0
2,1.0,NaN,5.0
3,NaN,Top Floor,3.0
4,-1.0,Garden Floor,3.0
5,-1.0,Garden Floor,3.0
6,NaN,Top Floor,3.0
7,-1.0,Garden Floor,4.0
8,3.0,NaN,3.0
9,-1.0,Garden Floor,2.0


In [20]:
df_clean["floor_category"].value_counts(dropna=False)

floor_category
NaN                  17364
Garden Floor          1838
Raised Ground         1765
Entrance Floor        1009
Mid Floor              954
Top Floor              433
Sub-level 1            292
Penthouse              265
21 and Above           188
Ground Floor           181
Basement               125
Semi-Basement          107
Sub-level 2            106
Terrace Floor           40
Villa Floor             35
Sub-level 3             29
Basement & Ground       22
Name: count, dtype: int64

In [21]:
df_clean.groupby("is_in_complex")[
    "maintenance_fee"
].apply(
    lambda x: pd.Series({
        "missing_percent": x.isna().mean() * 100,
        "median": x.median(),
        "count": x.notna().sum()
    })
)

is_in_complex                 
0              missing_percent      64.537456
               median              250.000000
               count              6812.000000
1              missing_percent      60.588023
               median             2926.000000
               count              2185.000000
Name: maintenance_fee, dtype: float64

categorical_columns = [
    "building_type",
    "building_condition",
    "fuel_type",
    "orientation",
    "exchange",
    "furnished",
    "usage_status",
    "credit_eligible",
    "deed_status"
]

for col in categorical_columns:
    print(f"\n--- {col} ---")
    print(df_clean[col].value_counts(dropna=False))

Categorical Data Cleaning

In [22]:
df_clean.loc[
    df_clean["orientation"] == "Fourplex",
    "orientation"
] = np.nan

In [23]:
df_clean["orientation"].value_counts(dropna=False)

orientation
NaN                         5311
South                       4680
South, East                 3380
South, East, West           2418
North, South, East, West    2025
South, West                 1533
North, South                 956
East                         836
East, West                   800
West                         544
North, South, East           447
North, East                  440
North, West                  409
North                        360
North, East, West            307
North, South, West           307
Name: count, dtype: int64

In [24]:
directions = ["North", "South", "East", "West"]

for direction in directions:
    df_clean[f"orientation_{direction.lower()}"] = (
        df_clean["orientation"]
        .fillna("")
        .str.contains(direction)
        .astype(int)
    )

In [25]:
df_clean[
    [
        "orientation",
        "orientation_north",
        "orientation_south",
        "orientation_east",
        "orientation_west"
    ]
].head(15)

,orientation,orientation_north,orientation_south,orientation_east,orientation_west
0,"North, South, East, West",1,1,1,1
1,"North, South, East, West",1,1,1,1
2,"North, East, West",1,0,1,1
3,"North, West",1,0,0,1
4,NaN,0,0,0,0
5,North,1,0,0,0
6,"South, East, West",0,1,1,1
7,"North, South, East",1,1,1,0
8,"North, South, East, West",1,1,1,1
9,"North, South, East, West",1,1,1,1


Floor Feature Engineering

In [26]:
df_clean["is_top_floor"] = (
    df_clean["floor_category"]
    .fillna("")
    .isin(["Top Floor", "Penthouse", "Terrace Floor"])
    .astype(int)
)

df_clean["is_ground_floor"] = (
    df_clean["floor_category"]
    .fillna("")
    .isin([
        "Garden Floor",
        "Raised Ground",
        "Entrance Floor",
        "Ground Floor"
    ])
    .astype(int)
)

df_clean["is_basement"] = (
    df_clean["floor_category"]
    .fillna("")
    .str.contains(
        "Basement|Sub-Level",
        regex=True
    )
    .astype(int)
)

In [27]:
df_clean[
    [
        "floor",
        "floor_category",
        "is_top_floor",
        "is_ground_floor",
        "is_basement"
    ]
].head(30)

,floor,floor_category,is_top_floor,is_ground_floor,is_basement
0,3.0,NaN,0,0,0
1,-1.0,Garden Floor,0,1,0
2,1.0,NaN,0,0,0
3,NaN,Top Floor,1,0,0
4,-1.0,Garden Floor,0,1,0
5,-1.0,Garden Floor,0,1,0
6,NaN,Top Floor,1,0,0
7,-1.0,Garden Floor,0,1,0
8,3.0,NaN,0,0,0
9,-1.0,Garden Floor,0,1,0


Categorical Data Cleaning

In [28]:
df_clean.loc[
    df_clean["orientation"] == "Fourplex",
    "orientation"
] = np.nan

In [29]:
df_clean.loc[
    df_clean["orientation"] == "Fourplex",
    "orientation"
] = np.nan

In [30]:
df_clean.loc[
    df_clean["orientation"] == "Fourplex",
    "orientation"
] = np.nan

In [31]:
df_clean["orientation"].value_counts(dropna=False)

orientation
NaN                         5311
South                       4680
South, East                 3380
South, East, West           2418
North, South, East, West    2025
South, West                 1533
North, South                 956
East                         836
East, West                   800
West                         544
North, South, East           447
North, East                  440
North, West                  409
North                        360
North, East, West            307
North, South, West           307
Name: count, dtype: int64

In [32]:
directions = ["North", "South", "East", "West"]

for direction in directions:
    df_clean[f"orientation_{direction.lower()}"] = (
        df_clean["orientation"]
        .fillna("")
        .str.contains(direction)
        .astype(int)
    )

In [33]:
df_clean[
    [
        "orientation",
        "orientation_north",
        "orientation_south",
        "orientation_east",
        "orientation_west"
    ]
].head(15)

,orientation,orientation_north,orientation_south,orientation_east,orientation_west
0,"North, South, East, West",1,1,1,1
1,"North, South, East, West",1,1,1,1
2,"North, East, West",1,0,1,1
3,"North, West",1,0,0,1
4,NaN,0,0,0,0
5,North,1,0,0,0
6,"South, East, West",0,1,1,1
7,"North, South, East",1,1,1,0
8,"North, South, East, West",1,1,1,1
9,"North, South, East, West",1,1,1,1


Floor Feature Engineering

In [34]:
df_clean["is_top_floor"] = (
    df_clean["floor_category"]
    .fillna("")
    .isin(["Top Floor", "Penthouse", "Terrace Floor"])
    .astype(int)
)

df_clean["is_ground_floor"] = (
    df_clean["floor_category"]
    .fillna("")
    .isin([
        "Garden Floor",
        "Raised Ground",
        "Entrance Floor",
        "Ground Floor"
    ])
    .astype(int)
)

df_clean["is_basement"] = (
    df_clean["floor_category"]
    .fillna("")
    .str.contains(
        "Basement|Sub-Level",
        regex=True
    )
    .astype(int)
)

In [35]:
df_clean[
    [
        "floor",
        "floor_category",
        "is_top_floor",
        "is_ground_floor",
        "is_basement"
    ]
].head(30)

,floor,floor_category,is_top_floor,is_ground_floor,is_basement
0,3.0,NaN,0,0,0
1,-1.0,Garden Floor,0,1,0
2,1.0,NaN,0,0,0
3,NaN,Top Floor,1,0,0
4,-1.0,Garden Floor,0,1,0
5,-1.0,Garden Floor,0,1,0
6,NaN,Top Floor,1,0,0
7,-1.0,Garden Floor,0,1,0
8,3.0,NaN,0,0,0
9,-1.0,Garden Floor,0,1,0


In [36]:
print("Missing floor before:",
      df_clean["floor"].isna().sum())

Missing floor before: 1915


In [37]:
df_clean.loc[
    df_clean["floor"].isna()
    & (df_clean["floor_category"] == "Top Floor"),
    "floor"
] = df_clean["total_floors"]

df_clean.loc[
    df_clean["floor"].isna()
    & (df_clean["floor_category"] == "Penthouse"),
    "floor"
] = df_clean["total_floors"]

df_clean.loc[
    df_clean["floor"].isna()
    & (df_clean["floor_category"] == "Terrace Floor"),
    "floor"
] = df_clean["total_floors"]

In [38]:
print("Missing floor after rule-based fill:",
      df_clean["floor"].isna().sum())

Missing floor after rule-based fill: 1177


In [39]:
df_clean["floor_ratio"] = (
    df_clean["floor"]
    / df_clean["total_floors"]
)

In [40]:
invalid_floor_ratio = (
    (df_clean["floor_ratio"] < -1)
    |
    (df_clean["floor_ratio"] > 1.5)
)

print("Invalid floor ratios:",
      invalid_floor_ratio.sum())

df_clean.loc[
    invalid_floor_ratio,
    "floor_ratio"
] = np.nan

Invalid floor ratios: 17


In [41]:
df_clean[
    [
        "floor",
        "total_floors",
        "floor_category",
        "floor_ratio"
    ]
].head(20)

,floor,total_floors,floor_category,floor_ratio
0,3.0,3.0,NaN,1.000000
1,-1.0,3.0,Garden Floor,-0.333333
2,1.0,5.0,NaN,0.200000
3,3.0,3.0,Top Floor,1.000000
4,-1.0,3.0,Garden Floor,-0.333333
5,-1.0,3.0,Garden Floor,-0.333333
6,3.0,3.0,Top Floor,1.000000
7,-1.0,4.0,Garden Floor,-0.250000
8,3.0,3.0,NaN,1.000000
9,-1.0,2.0,Garden Floor,-0.500000
